In [ ]:
# ============================================================
# TASK 2: CHUNKING, EMBEDDING, AND VECTOR STORE
# Complete runnable script – all steps in one cell
# ============================================================

# Step 1: Mount Drive & Install
from google.colab import drive
drive.mount('/content/drive')

!pip install sentence-transformers faiss-cpu -q

import pandas as pd
import numpy as np
import pickle
import os
import faiss
from sentence_transformers import SentenceTransformer
import torch
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")
print("GPU available:", torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

# Step 2: Load filtered data
print("\nLoading filtered data...")
file_path = '/content/drive/MyDrive/rag-complaint-chatbot/data/processed/filtered_complaints.csv'

if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}\nPlease complete Task 1 first.")

df = pd.read_csv(file_path, usecols=['Product', 'cleaned_narrative'])
print(f"Loaded {len(df):,} records")

# Step 3: Stratified sample (2,000 complaints)
print("\nCreating stratified sample...")
sample_size = 2000
product_counts = df['Product'].value_counts()
proportions = product_counts / len(df)

sampled_list = []
for product, prop in proportions.items():
    n = max(1, int(sample_size * prop))
    product_subset = df[df['Product'] == product]
    n = min(n, len(product_subset))
    if n > 0:
        sampled_list.append(product_subset.sample(n, random_state=42))

sampled = pd.concat(sampled_list, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

if len(sampled) < sample_size:
    remaining = df.drop(sampled.index, errors='ignore')
    additional = remaining.sample(min(sample_size - len(sampled), len(remaining)), random_state=42)
    sampled = pd.concat([sampled, additional], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Sample size: {len(sampled):,}")
print(sampled['Product'].value_counts())

# Step 4: Chunking function
def chunk_text(text, chunk_size=500, overlap=50):
    if not text or len(text) <= chunk_size:
        return [text] if text else []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            for sep in ['. ', '! ', '? ', '.', '!', '?']:
                pos = text.rfind(sep, start, end)
                if pos != -1:
                    end = pos + len(sep)
                    break
        chunk = text[start:end].strip()
        if len(chunk) >= 50:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Step 5: Apply chunking to sample
print("\nChunking narratives...")
chunked_data = []
for idx, row in sampled.iterrows():
    chunks = chunk_text(row['cleaned_narrative'])
    for ci, chunk in enumerate(chunks):
        chunked_data.append({
            'complaint_id': idx,
            'product_category': row['Product'],
            'chunk_index': ci,
            'chunk_text': chunk,
            'chunk_length': len(chunk)
        })
print(f"Created {len(chunked_data):,} chunks")

# Step 6: Generate embeddings
texts = [c['chunk_text'] for c in chunked_data]
print("\nLoading SentenceTransformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print("Generating embeddings...")
embeddings = model.encode(
    texts,
    batch_size=64 if device == 'cuda' else 16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
print(f"Embeddings shape: {embeddings.shape}")

# Step 7: Build FAISS index
print("\nCreating FAISS index...")
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype(np.float32))
print(f"Index size: {index.ntotal:,} vectors")

# Step 8: Save to Google Drive
save_path = '/content/drive/MyDrive/rag-complaint-chatbot/vector_store/'
os.makedirs(save_path, exist_ok=True)

faiss.write_index(index, save_path + 'index.faiss')
with open(save_path + 'chunks.pkl', 'wb') as f:
    pickle.dump(chunked_data, f)
np.save(save_path + 'embeddings.npy', embeddings)

print(f"\nSaved to: {save_path}")
print("  - index.faiss")
print("  - chunks.pkl")
print("  - embeddings.npy")

# Step 9: Test retrieval
print("\nTesting retrieval...")
queries = [
    "Why are customers unhappy with credit cards?",
    "What issues do people have with money transfers?",
    "How do customers feel about personal loans?"
]

for q in queries:
    print(f"\nQuery: {q}")
    q_emb = model.encode([q], normalize_embeddings=True).astype(np.float32)
    distances, idxs = index.search(q_emb, k=3)
    for i, idx in enumerate(idxs[0]):
        if idx < len(chunked_data):
            print(f"  [{i+1}] Score={distances[0][i]:.4f}")
            print(f"      Product: {chunked_data[idx]['product_category']}")
            print(f"      Text: {chunked_data[idx]['chunk_text'][:120]}...")

# Step 10: Statistics
print("\n" + "="*60)
print("VECTOR STORE STATISTICS")
print("="*60)
print(f"Total vectors: {index.ntotal:,}")
print(f"Embedding dimension: {index.d}")
print(f"Total chunks: {len(chunked_data):,}")
print(f"Sample size: {len(sampled):,}")
print(f"Model: all-MiniLM-L6-v2")
print("\nTask 2 Complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.
GPU available: False
Using device: cpu

Loading filtered data...
Loaded 447,588 records

Creating stratified sample...
Sample size: 2,000
Product
Checking or savings account                                616
Credit card or prepaid card                                477
Money transfer, virtual currency, or money service         431
Credit card                                                356
Payday loan, title loan, or personal loan                   75
Payday loan, title loan, personal loan, or advance loan     39
Money transfers                                              6
Name: count, dtype: int64

Chunking narratives...
Created 5,291 chunks

Loading SentenceTransformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...


Batches:   0%|          | 0/331 [00:00<?, ?it/s]

Embeddings shape: (5291, 384)

Creating FAISS index...
Index size: 5,291 vectors

Saved to: /content/drive/MyDrive/rag-complaint-chatbot/vector_store/
  - index.faiss
  - chunks.pkl
  - embeddings.npy

Testing retrieval...

Query: Why are customers unhappy with credit cards?
  [1] Score=0.5885
      Product: Credit card or prepaid card
      Text: he bad credit debt to all 3 major credit bureau and is currently reflecting on my credit report of payment owed of 85.00...
  [2] Score=0.5822
      Product: Credit card
      Text: t had any of the cards go into default, and they continue to send me offers to bring over my balances and investments th...
  [3] Score=0.5695
      Product: Credit card
      Text: y, or the postal office, or another reputable bank in my town, all offers we denied. i feel these practices are predator...

Query: What issues do people have with money transfers?
  [1] Score=0.5544
      Product: Checking or savings account
      Text: availability of funds and have 

In [ ]:
from google.colab import files
import os

save_path = '/content/drive/MyDrive/rag-complaint-chatbot/vector_store/'

for fname in ['index.faiss', 'chunks.pkl', 'embeddings.npy']:
    file_path = os.path.join(save_path, fname)
    if os.path.exists(file_path):
        files.download(file_path)
        print(f"Downloaded: {fname}")
    else:
        print(f"File not found: {fname}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: index.faiss


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: chunks.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: embeddings.npy


In [ ]:
import os
drive_path = '/content/drive/MyDrive/rag-complaint-chatbot/vector_store/'
if os.path.exists(drive_path):
    print("Files in vector_store:")
    for f in os.listdir(drive_path):
        size = os.path.getsize(drive_path + f) / (1024*1024)
        print(f"  - {f} ({size:.2f} MB)")
else:
    print("vector_store folder not found at that path.")

Files in vector_store:
  - index.faiss (7.75 MB)
  - chunks.pkl (1.96 MB)
  - embeddings.npy (7.75 MB)


In [ ]:
import os
import shutil
from google.colab import files

# Source: Drive
drive_path = '/content/drive/MyDrive/rag-complaint-chatbot/vector_store/'

# Destination: Local Colab VM
local_path = '/content/downloads/'
os.makedirs(local_path, exist_ok=True)

# List files to download
file_list = ['index.faiss', 'chunks.pkl', 'embeddings.npy']

print("Copying files to local VM...")
for fname in file_list:
    src = os.path.join(drive_path, fname)
    dst = os.path.join(local_path, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size = os.path.getsize(dst) / (1024 * 1024)
        print(f" Copied: {fname} ({size:.2f} MB)")
    else:
        print(f"File not found: {fname}")

Copying files to local VM...
 Copied: index.faiss (7.75 MB)
 Copied: chunks.pkl (1.96 MB)
 Copied: embeddings.npy (7.75 MB)


In [ ]:
import os
local_path = '/content/downloads/'
print("Files ready for download:")
for f in os.listdir(local_path):
    size = os.path.getsize(os.path.join(local_path, f)) / (1024 * 1024)
    print(f"  - {f} ({size:.2f} MB)")

Files ready for download:
  - embeddings.npy (7.75 MB)
  - chunks.pkl (1.96 MB)
  - index.faiss (7.75 MB)


In [ ]:
from google.colab import files
import shutil
import os

drive_path = '/content/drive/MyDrive/rag-complaint-chatbot/vector_store/'
local_path = '/content/'

for fname in ['index.faiss', 'chunks.pkl', 'embeddings.npy']:
    src = os.path.join(drive_path, fname)
    if os.path.exists(src):
        shutil.copy2(src, local_path + fname)
        files.download(local_path + fname)
        print(f"Downloaded: {fname}")
    else:
        print(f"Not found: {fname}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: index.faiss


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: chunks.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: embeddings.npy


In [ ]:
!zip -r /content/vector_store.zip /content/drive/MyDrive/rag-complaint-chatbot/vector_store/
from google.colab import files
files.download('/content/vector_store.zip')

  adding: content/drive/MyDrive/rag-complaint-chatbot/vector_store/ (stored 0%)
  adding: content/drive/MyDrive/rag-complaint-chatbot/vector_store/index.faiss (deflated 8%)
  adding: content/drive/MyDrive/rag-complaint-chatbot/vector_store/chunks.pkl (deflated 68%)
  adding: content/drive/MyDrive/rag-complaint-chatbot/vector_store/embeddings.npy (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
print("Files in vector_store:")
path = '/content/drive/MyDrive/rag-complaint-chatbot/vector_store/'
if os.path.exists(path):
    for f in os.listdir(path):
        print(f"  - {f}")
else:
    print("Folder not found. Searching...")
    os.system('find /content/drive -name "index.faiss" 2>/dev/null')

Files in vector_store:
  - index.faiss
  - chunks.pkl
  - embeddings.npy


In [1]:
# ============================================================
# TASK 2: STRATIFIED SAMPLING SCRIPT
# ============================================================

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

print("="*60)
print("STRATIFIED SAMPLING")
print("="*60)

# Load the filtered data from Task 1
file_path = '/content/drive/MyDrive/rag-complaint-chatbot/data/processed/filtered_complaints.csv'
df = pd.read_csv(file_path)
print(f"Loaded {len(df):,} records")
print(f"Products in data: {df['Product'].unique().tolist()}")

# Define sample size (use 10,000 or adjust as needed)
sample_size = 10000

print(f"\nTarget sample size: {sample_size:,}")

# Stratified sampling by product
stratified_sample = df.groupby('Product', group_keys=False).apply(
    lambda x: x.sample(
        n=min(len(x), int(sample_size * len(x) / len(df))),
        random_state=42
    )
)

# If we have fewer than sample_size, add random records
if len(stratified_sample) < sample_size:
    remaining = df.drop(stratified_sample.index)
    additional = remaining.sample(sample_size - len(stratified_sample), random_state=42)
    stratified_sample = pd.concat([stratified_sample, additional])

# Shuffle
stratified_sample = stratified_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nActual sample size: {len(stratified_sample):,}")
print("\nSample distribution by product:")
print(stratified_sample['Product'].value_counts())

# Save the sample
save_path = '/content/drive/MyDrive/rag-complaint-chatbot/data/processed/stratified_sample.csv'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
stratified_sample.to_csv(save_path, index=False)
print(f"\n Saved: {save_path}")

# Show sample of what was saved
print("\nSample preview:")
print(stratified_sample[['Product', 'Consumer complaint narrative']].head(3))

Mounted at /content/drive
STRATIFIED SAMPLING
Loaded 447,588 records
Products in data: ['Credit card', 'Checking or savings account', 'Payday loan, title loan, personal loan, or advance loan', 'Money transfer, virtual currency, or money service', 'Credit card or prepaid card', 'Money transfers', 'Payday loan, title loan, or personal loan']

Target sample size: 10,000

Actual sample size: 10,000

Sample distribution by product:
Product
Checking or savings account                                3079
Credit card or prepaid card                                2386
Money transfer, virtual currency, or money service         2148
Credit card                                                1781
Payday loan, title loan, or personal loan                   378
Payday loan, title loan, personal loan, or advance loan     195
Money transfers                                              33
Name: count, dtype: int64

 Saved: /content/drive/MyDrive/rag-complaint-chatbot/data/processed/stratified_sample.

In [2]:
# ============================================================
# TASK 2: CLEAR STRATIFIED SAMPLING SCRIPT
# ============================================================

import pandas as pd

print("="*60)
print("STRATIFIED SAMPLING")
print("="*60)

# Load the filtered data from Task 1
file_path = '/content/drive/MyDrive/rag-complaint-chatbot/data/processed/filtered_complaints.csv'
df = pd.read_csv(file_path)
print(f"Loaded {len(df):,} records")
print(f"Products in data: {df['Product'].unique().tolist()}")

# Define sample size
sample_size = 10000

print(f"\nTarget sample size: {sample_size:,}")

# Stratified sampling by product
stratified_sample = df.groupby('Product', group_keys=False).apply(
    lambda x: x.sample(
        n=min(len(x), int(sample_size * len(x) / len(df))),
        random_state=42
    )
)

# If we have fewer than sample_size, add random records
if len(stratified_sample) < sample_size:
    remaining = df.drop(stratified_sample.index)
    additional = remaining.sample(sample_size - len(stratified_sample), random_state=42)
    stratified_sample = pd.concat([stratified_sample, additional])

# Shuffle
stratified_sample = stratified_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nActual sample size: {len(stratified_sample):,}")
print("\nSample distribution by product:")
print(stratified_sample['Product'].value_counts())

# Save the sample
save_path = '/content/drive/MyDrive/rag-complaint-chatbot/data/processed/stratified_sample.csv'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
stratified_sample.to_csv(save_path, index=False)
print(f"\n Saved: {save_path}")

# Show preview
print("\nSample preview:")
print(stratified_sample[['Product', 'Consumer complaint narrative']].head(3))

STRATIFIED SAMPLING
Loaded 447,588 records
Products in data: ['Credit card', 'Checking or savings account', 'Payday loan, title loan, personal loan, or advance loan', 'Money transfer, virtual currency, or money service', 'Credit card or prepaid card', 'Money transfers', 'Payday loan, title loan, or personal loan']

Target sample size: 10,000

Actual sample size: 10,000

Sample distribution by product:
Product
Checking or savings account                                3079
Credit card or prepaid card                                2386
Money transfer, virtual currency, or money service         2148
Credit card                                                1781
Payday loan, title loan, or personal loan                   378
Payday loan, title loan, personal loan, or advance loan     195
Money transfers                                              33
Name: count, dtype: int64

✅ Saved: /content/drive/MyDrive/rag-complaint-chatbot/data/processed/stratified_sample.csv

Sample preview:
    